# G6M2: Surface (Global Time-Varying) vs Baseline (I²)

Compare two voltage degradation models on the G6M2 dataset:

| Model | Fitting Strategy | Description |
|-------|-----------------|-------------|
| **Baseline (I²)** | Per-interval sliding-window OLS | 5 parameters, independent fits per window |
| **Surface** | Global single fit (all data) | 10 parameters, ci(t) = ai·t + bi time-varying |

Model equation (Surface):
$$U(I,T,h,t) = c_1(t)\cdot I_s + c_2(t)\cdot (IT)_s + c_3(t)\cdot \ln(h)_s + c_4(t)\cdot I^2_s + c_5(t) + U_{OCV}(T)$$
where $c_i(t) = a_i \cdot t + b_i$, optimizer: **scipy least_squares (Levenberg-Marquardt)**.

SE is computed via: $\text{SE} = \sqrt{\sigma^2 (1 + x_0^T (X^TX)^{-1} x_0)} \cdot \text{scale}$
(aligned with Urc1 XTX/sigma2 variance propagation method)

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pathlib import Path
from datetime import datetime

from degradation_toolbox.Urc.Urc1 import Urc1
from degradation_toolbox.Urc.Urc1_surface import Urc1_Surface
from master_arbeit_Di.explore.UnifiedModelComparator import UnifiedModelComparator
from master_arbeit_Di.explore.GMpreprocess import GMpreprocess

print("All imports successful")
print(f"Current directory: {os.getcwd()}")

All imports successful
Current directory: c:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\surface


In [2]:
now = datetime.now()
print("=" * 80)
print("Training start time:", now.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

Training start time: 2026-05-04 20:59:28


In [3]:
# =============================================================================
# CONFIGURATION
# =============================================================================

DATASET_PATH = r"..\..\explore_data\G6M2.parquet"
PREPROCESS_OUTPUT_DIR = r"..\..\explore_data\output"
PLOTS_OUTPUT_DIR = r"..\plots\surface\G6M2_comparison"

os.makedirs(PLOTS_OUTPUT_DIR, exist_ok=True)

# Reference condition configurations: Low, Medium, High
REF_CONFIGS = {
    "Low": {
        "Iref": 0.28,
        "Tref": 57,
        "OHref": 10,
        "gt_file": r"..\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_regression_full_coverage.csv",
    },
    "Medium": {
        "Iref": 1.0,
        "Tref": 58,
        "OHref": 33,
        "gt_file": r"..\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_regression_full_coverage.csv",
    },
    "High": {
        "Iref": 1.31,
        "Tref": 58,
        "OHref": 18,
        "gt_file": r"..\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_regression_full_coverage.csv",
    },
}

COMMON_CONFIG = {
    "Iref": [cfg["Iref"] for cfg in REF_CONFIGS.values()],
    "Tref": 60,
    "OHref": 72,
    "ref_config": REF_CONFIGS,
    "len_interval": 2,
    "slide": 1,
    "min_num_data_required_for_fit": 300,
    "threshold": 1e6,
    "i_off": 0.1,
    "u_off": 1.3,
    "plot_fit": 0,
    "data_filter_i_min": 0.1,
    "data_filter_U_min": 1.4,
    "data_filter_U_max": 2.3,
    "data_filter_T_min": 50,
    "data_filter_T_max": 65,
}

# Surface-specific parameter
TRAIN_RATIO = 0.8

SHOW_GT_METRICS = True
SHOW_ALL_COND_METRICS = True

print("Configuration loaded successfully")
print(f"  Reference conditions: {list(REF_CONFIGS.keys())}")
print(f"  Train ratio (Surface): {TRAIN_RATIO}")
print(f"  Output directory: {PLOTS_OUTPUT_DIR}")

Configuration loaded successfully
  Reference conditions: ['Low', 'Medium', 'High']
  Train ratio (Surface): 0.8
  Output directory: ..\plots\surface\G6M2_comparison


In [4]:
# =============================================================================
# STEP 1: DATA LOADING & PREPROCESSING
# =============================================================================
print("=" * 80)
print("STEP 1: Data Loading & Preprocessing")
print("=" * 80)

preprocessor = GMpreprocess(file_path=DATASET_PATH, output_dir=PREPROCESS_OUTPUT_DIR)
data = preprocessor.run()
dataset_name = preprocessor.name

print(f"\nDataset: {dataset_name}")
print(f"  Shape: {data.shape}")
print(f"  Time range: {data.index.min()} -> {data.index.max()}")

# Preprocess once and reuse across all models
shared_pre = Urc1.preprocess_once(
    data,
    i_off=COMMON_CONFIG["i_off"],
    u_off=COMMON_CONFIG["u_off"],
    data_filter_i_min=COMMON_CONFIG["data_filter_i_min"],
    data_filter_U_min=COMMON_CONFIG["data_filter_U_min"],
    data_filter_U_max=COMMON_CONFIG["data_filter_U_max"],
    data_filter_T_min=COMMON_CONFIG["data_filter_T_min"],
    data_filter_T_max=COMMON_CONFIG["data_filter_T_max"],
)
print(f"\nShared preprocessed data: {len(shared_pre)} rows")

STEP 1: Data Loading & Preprocessing
=== 1. Loading & Preprocessing: G6M2 ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_12' -> ID: '2'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   ..\..\explore_data\output\G6M2_20260504_205929.parquet

=== GMpreprocess Pipeline Completed Successfully ===

Dataset: G6M2
  Shape: (1096020, 3)
  Time range: 2023-07-06 00:00:00 -> 2025-08-05 09:29:00
[preprocess_once] 1096020 -> 359353 points.

Shared preprocessed data: 359353 rows


In [5]:
# =============================================================================
# STEP 2: TRAIN MODELS
# =============================================================================
print("\n" + "=" * 80)
print("STEP 2: Model Training (Baseline I² | Surface)")
print("=" * 80)

models = {}

# --- Baseline (I²) ---
print("\n  --> Training Baseline (I²) model...")
t_start = datetime.now()
try:
    model_baseline = Urc1(
        data=data,
        name=dataset_name,
        preprocessed_data=shared_pre,
        **COMMON_CONFIG,
    )
    models["Baseline (I²)"] = model_baseline
    print(f"    Training time: {(datetime.now() - t_start).total_seconds():.1f}s")
except Exception as e:
    print(f"    Training failed: {e}")

# --- Surface ---
print("\n  --> Training Surface (Global Time-Varying) model...")
t_start = datetime.now()
try:
    model_surface = Urc1_Surface(
        data=data,
        name=dataset_name,
        preprocessed_data=shared_pre,
        train_ratio=TRAIN_RATIO,
        **COMMON_CONFIG,
    )
    models["Surface"] = model_surface
    print(f"    Training time: {(datetime.now() - t_start).total_seconds():.1f}s")
    print(f"    Test RMSE: {model_surface.test_metrics.get('test_rmse_mv', float('nan')):.2f} mV")
    print(f"    Test R²:   {model_surface.test_metrics.get('test_r2', float('nan')):.4f}")
except Exception as e:
    print(f"    Training failed: {e}")
    import traceback; traceback.print_exc()

print(f"\nSuccessfully trained {len(models)}/2 models: {list(models.keys())}")


STEP 2: Model Training (Baseline I² | Surface)

  --> Training Baseline (I²) model...
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Fitting Stats: 218 intervals low data, 0 fit failed.
468 out of 762 fitting results are reliable.
    Training time: 6.3s

  --> Training Surface (Global Time-Varying) model...
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...

Starting Global Surface Fitting (Train: 80%)...
   Train: 287482 points | Test: 71871 points
   Optimization converged (cost: 1.7181e+00)
   Message: `gtol` termination condition is satisfied.
   status: 1
758 out of 758 fitting results are reliable.

Urc1_Surface Initialized:
   Train Ratio: 80%
   Test RMSE (Point): 6.06 mV
    Training time: 5.2s
    Test RMSE: 6.06 mV
    Test R²:   0.9947

Successfully trained 2/2 models: ['Baseline (I²)', 'Surface']


In [6]:
now = datetime.now()
print("=" * 80)
print("Training finished time:", now.strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 80)

Training finished time: 2026-05-04 20:59:51


In [7]:
# =============================================================================
# STEP 3: COMPARATOR SETUP & GROUND TRUTH LOADING
# =============================================================================
print("\n" + "=" * 80)
print("STEP 3: Initialize Comparator & Load Ground Truth")
print("=" * 80)

comparator = UnifiedModelComparator(models)
print(f"UnifiedModelComparator initialized with {len(models)} models")

try:
    project_root = Path(__file__).parent.parent
except NameError:
    project_root = Path.cwd()

print(f"Project root: {project_root}\n")

gt_loaded_count = 0
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    gt_path = Path(ref_cfg["gt_file"])
    if not gt_path.is_absolute():
        gt_path = (project_root / gt_path).resolve()

    print(f"  Looking for GT file: {gt_path}")

    if gt_path.exists():
        try:
            gt_data = pd.read_csv(gt_path, index_col=0, parse_dates=True)
            colmap = {str(c).strip().lower(): c for c in gt_data.columns}
            candidate_cols = ["gt_uref_regression", "voltage", "uref", "gt_uref"]
            selected_col = next((colmap[c] for c in candidate_cols if c in colmap), None)
            if selected_col is None and len(gt_data.columns) == 1:
                selected_col = gt_data.columns[0]
            if selected_col is None:
                raise ValueError(f"Cannot identify voltage column in {gt_path}")
            gt_series = gt_data[selected_col].dropna()
            comparator.set_ground_truth(gt_series, iref=iref)
            gt_loaded_count += 1
            print(f"  Loaded GT for {ref_name} (Iref={iref}): {len(gt_series)} points [col: {selected_col}]")
        except Exception as e:
            print(f"  Failed to load GT for {ref_name}: {e}")
    else:
        print(f"  GT file NOT found: {gt_path}")

has_gt = gt_loaded_count > 0
print(f"\n{'='*80}")
print(f"GT status: {gt_loaded_count}/{len(REF_CONFIGS)} reference conditions loaded")
print(f"GT metrics will be {'ENABLED' if has_gt else 'DISABLED (no GT files found)'}")
print(f"{'='*80}\n")


STEP 3: Initialize Comparator & Load Ground Truth
UnifiedModelComparator initialized with 2 models
Project root: c:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\surface

  Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_regression_full_coverage.csv
  Loaded GT for Low (Iref=0.28): 758 points [col: gt_uref_regression]
  Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_regression_full_coverage.csv
  Loaded GT for Medium (Iref=1.0): 758 points [col: gt_uref_regression]
  Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_regression_full_coverage.csv
  Loaded GT for High (Iref=1.3

In [8]:
# =============================================================================
# STEP 4: REFERENCE-SPECIFIC ANALYSIS & METRICS
# =============================================================================
print("=" * 80)
print("STEP 4: Reference-Specific Metrics & Comparison")
print("=" * 80)

rate_tables = []

for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    tref = ref_cfg["Tref"]
    ohref = ref_cfg["OHref"]

    print(f"\nREFERENCE CONDITION: {ref_name} (Iref={iref}, Tref={tref}, OHref={ohref}) ---")

    # Metrics table
    df_metrics = comparator.compare_all(
        i_target=iref,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
        include_gt_metrics=has_gt,
    )
    print(f"\nPerformance Metrics:")
    print(df_metrics.to_markdown(index=False))

    rate_tables.append(
        df_metrics[["Model Name", "Target Current (A/cm2)", "Degradation Rate (uV/h)", "Slope Sigma (uV/h)"]].assign(
            Reference=ref_name
        )
    )

    # Trend plot
    try:
        comparator.plot_interactive_trends(
            target_i=iref,
            show_gt=has_gt,
            save=True,
            output_dir=PLOTS_OUTPUT_DIR,
            uncertainty_style="band",
            uncertainty_opacity=0.12,
            show_series_line=True,
            rate_precision=6,
        )
        print(f"Trend plot saved for {ref_name}")
    except Exception as e:
        print(f"Trend plot failed for {ref_name}: {e}")

    # Detailed report
    print(f"\nDetailed Comparison Report:")
    comparator.print_comparison_report(
        i_target=iref,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
        include_gt_metrics=has_gt,
    )

STEP 4: Reference-Specific Metrics & Comparison

REFERENCE CONDITION: Low (Iref=0.28, Tref=57, OHref=10) ---

Performance Metrics:
| Model Name    |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:--------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline (I²) |                     0.28 |              6.191 |               468 |            

In [9]:
# Degradation rate summary across all reference conditions
if rate_tables:
    summary_df = pd.concat(rate_tables, ignore_index=True)
    print("\nDegradation Rate Summary (all reference conditions):")
    print(summary_df.to_markdown(index=False))


Degradation Rate Summary (all reference conditions):
| Model Name    |   Target Current (A/cm2) |   Degradation Rate (uV/h) |   Slope Sigma (uV/h) | Reference   |
|:--------------|-------------------------:|--------------------------:|---------------------:|:------------|
| Baseline (I²) |                     0.28 |                   2.15303 |                0     | Low         |
| Surface       |                     0.28 |                   2.39019 |                0.007 | Low         |
| Baseline (I²) |                     1    |                   5.84556 |                0     | Medium      |
| Surface       |                     1    |                   4.90876 |                0.199 | Medium      |
| Baseline (I²) |                     1.31 |                   8.46857 |                0     | High        |
| Surface       |                     1.31 |                   6.92786 |                0.287 | High        |


In [10]:
# =============================================================================
# STEP 5: CROSS-MODEL DIAGNOSTICS
# =============================================================================
print("\n" + "=" * 80)
print("STEP 5: Cross-Model Diagnostics")
print("=" * 80)

try:
    print("\nPlotting fit quality (RMSE & R2 distributions)...")
    comparator.plot_fit_quality(save=True, output_dir=PLOTS_OUTPUT_DIR)
    print("Fit quality plot saved")
except Exception as e:
    print(f"Fit quality plot failed: {e}")

try:
    print("\nPlotting coefficient diagnostics...")
    comparator.plot_coefficient_diagnostic(save=True, output_dir=PLOTS_OUTPUT_DIR)
    print("Coefficient diagnostic plot saved")
except Exception as e:
    print(f"Coefficient diagnostic plot failed: {e}")

try:
    print("\nPlotting coverage Gantt...")
    comparator.plot_coverage_gantt(save=True, output_dir=PLOTS_OUTPUT_DIR)
    print("Coverage Gantt saved")
except Exception as e:
    print(f"Coverage Gantt failed: {e}")


STEP 5: Cross-Model Diagnostics

Plotting fit quality (RMSE & R2 distributions)...
Fit quality plot saved

Plotting coefficient diagnostics...
Coefficient diagnostic plot saved

Plotting coverage Gantt...
Coverage Gantt saved


In [11]:
# =============================================================================
# STEP 6: SURFACE MODEL METRICS
# =============================================================================
print("\n" + "=" * 80)
print("STEP 6: Surface Model Specific Metrics")
print("=" * 80)

if "Surface" in models:
    surf = models["Surface"]
    print(f"\nGlobal Fit Quality (test set):")
    metrics = surf.test_metrics
    print(f"  Test RMSE:      {metrics.get('test_rmse_mv', float('nan')):.3f} mV")
    print(f"  Test R²:        {metrics.get('test_r2', float('nan')):.4f}")
    print(f"  Test MAE:       {metrics.get('test_mae_mv', float('nan')):.3f} mV")
    print(f"  Test MaxError:  {metrics.get('test_max_error_mv', float('nan')):.3f} mV")
    print(f"  N test points:  {metrics.get('n_test_points', 'N/A')}")

    print(f"\nTime-varying parameter slopes (degradation rates per 1000h):")
    a = surf.coeffs[0::2]
    a_se = surf.coeffs_se[0::2] if surf.coeffs_se is not None else [float('nan')] * 5
    for j, (ai, sei) in enumerate(zip(a, a_se)):
        print(f"  a{j+1} = {ai:.4e}  ±  {sei:.2e}  (c{j+1} slope)")

    print(f"\nSE method: XTX/sigma2 variance propagation")
    print(f"  global_sigma2 = {surf.global_sigma2:.6f}" if surf.global_sigma2 is not None else "  global_sigma2 = None")
    print(f"  global_xtx shape = {surf.global_xtx.shape}" if surf.global_xtx is not None else "  global_xtx = None")


STEP 6: Surface Model Specific Metrics

Global Fit Quality (test set):
  Test RMSE:      6.057 mV
  Test R²:        0.9947
  Test MAE:       5.031 mV
  Test MaxError:  155.253 mV
  N test points:  71871

Time-varying parameter slopes (degradation rates per 1000h):
  a1 = 4.8567e-06  ±  2.51e-07  (c1 slope)
  a2 = -3.5630e-06  ±  2.62e-07  (c2 slope)
  a3 = -1.3540e-06  ±  7.48e-09  (c3 slope)
  a4 = 4.5051e-06  ±  7.38e-08  (c4 slope)
  a5 = 2.8961e-06  ±  4.37e-09  (c5 slope)

SE method: XTX/sigma2 variance propagation
  global_sigma2 = 0.000032
  global_xtx shape = (5, 5)


In [12]:
# =============================================================================
# STEP 7: CUSTOM REFERENCE VOLTAGE EXTRACTION
# =============================================================================
print("\n" + "=" * 80)
print("STEP 7: Custom Reference Voltage Extraction")
print("=" * 80)

ref_list = [
    {"Iref": ref_cfg["Iref"], "Tref": ref_cfg["Tref"], "OHref": ref_cfg["OHref"], "name": ref_name}
    for ref_name, ref_cfg in REF_CONFIGS.items()
]

for model_name, model in models.items():
    print(f"\nCalculating custom references for {model_name}...")
    try:
        urc_dict = model.calculate_urc_for_custom_refs(ref_list)
        print(f"  Extracted Urc for {len(urc_dict)} reference configurations:")
        for ref_name, urc_df in urc_dict.items():
            print(f"    - {ref_name}: {len(urc_df)} time points")
    except Exception as e:
        print(f"  Failed: {e}")


STEP 7: Custom Reference Voltage Extraction

Calculating custom references for Baseline (I²)...
Calculated Urc for Low: 468 points
Calculated Urc for Medium: 468 points
Calculated Urc for High: 468 points
  Extracted Urc for 3 reference configurations:
    - Low: 468 time points
    - Medium: 468 time points
    - High: 468 time points

Calculating custom references for Surface...
Calculated Urc for Low: 758 points
Calculated Urc for Medium: 758 points
Calculated Urc for High: 758 points
  Extracted Urc for 3 reference configurations:
    - Low: 758 time points
    - Medium: 758 time points
    - High: 758 time points


In [13]:
# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"\nTrained models: {list(models.keys())}")
print(f"Reference conditions analyzed: {list(REF_CONFIGS.keys())}")
print(f"Ground truth data: {'Loaded' if has_gt else 'Not available'}")
print(f"Output plots saved to: {PLOTS_OUTPUT_DIR}/")
print(f"\nModel notes:")
print(f"  - Baseline (I²): Per-interval OLS with 5 parameters, sliding window")
print(f"  - Surface: Global fit, 10 parameters, ci(t) = ai*t + bi, least_squares")
print(f"  - Both models use XTX/sigma2 variance propagation for SE calculation")


ANALYSIS COMPLETE

Trained models: ['Baseline (I²)', 'Surface']
Reference conditions analyzed: ['Low', 'Medium', 'High']
Ground truth data: Loaded
Output plots saved to: ..\plots\surface\G6M2_comparison/

Model notes:
  - Baseline (I²): Per-interval OLS with 5 parameters, sliding window
  - Surface: Global fit, 10 parameters, ci(t) = ai*t + bi, least_squares
  - Both models use XTX/sigma2 variance propagation for SE calculation
